In [2]:
import pandas as pd
import numpy as np
print('The Libraries are imported successfully')

The Libraries are imported successfully


In [3]:
print('Loading the Dataset.....')
try:
  df = pd.read_csv('Telco-Customer-Churn.csv')
  print("Loaded from saved file")
except:
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    df = pd.read_csv(url)
    print("Loaded from URL")

Loading the Dataset.....
Loaded from saved file


In [4]:
print(df.head(3))
print(f'Rows {df.shape[0]}Customers')
print(f'columns{df.shape[1]}Features')


for i ,col in enumerate(df.columns):
  print(f'{i}.{col}')

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   
2          No          No              No  Month-to-month              Yes   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  E

In [5]:
print(df.dtypes)
#count values by datatypes
type_count=df.dtypes.value_counts()
for dtype,count in type_count.items():
  print(f'{dtype}:{count}')

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
object:18
int64:2
float64:1


In [6]:
cat_cols=df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
  print(f'-{col}')

cat_nums=df.select_dtypes(include=['int64','float64']).columns.tolist()
for num in cat_nums:
  print(f'*{num}')

-customerID
-gender
-Partner
-Dependents
-PhoneService
-MultipleLines
-InternetService
-OnlineSecurity
-OnlineBackup
-DeviceProtection
-TechSupport
-StreamingTV
-StreamingMovies
-Contract
-PaperlessBilling
-PaymentMethod
-TotalCharges
-Churn
*SeniorCitizen
*tenure
*MonthlyCharges


In [7]:
#statistics
#For Specific Column
print(f'minimum:{df['tenure'].min()}months')
print(f'Maximum:{df['tenure'].max()}months')
print(f'Average:{df['tenure'].mean():2f}months')
print(f'Median:{df['tenure'].median()}month')

#for Monthly Charges
print(f'Minimum:{df['MonthlyCharges'].min()}Charges')
print(f'MAximum:{df['MonthlyCharges'].max()}Charges')
print(f'Average:{df['MonthlyCharges'].mean()}Charges')

minimum:0months
Maximum:72months
Average:32.371149months
Median:29.0month
Minimum:18.25Charges
MAximum:118.75Charges
Average:64.76169246059918Charges


In [8]:
#looking For Target Variable
churn_count=df['Churn'].value_counts()
churn_percentage=df['Churn'].value_counts(normalize=True)*100

print(f'churned:{churn_count['Yes']}Customers')
print(f'Non Churned:{churn_count['No']} Customers')

print(f'Churned Percentage:{churn_percentage['Yes']:.2f}%')
print(f'Non Churned Percentage:{churn_percentage['No']:.2f}%')

print(f"Note: We have class imbalance!")
print(f'For Every Churned Customer we have {churn_count['No']/churn_count['Yes']:.1f}Non Churned Customers')
print(f"We'll need to handle this when building our model")

churned:1869Customers
Non Churned:5174 Customers
Churned Percentage:26.54%
Non Churned Percentage:73.46%
Note: We have class imbalance!
For Every Churned Customer we have 2.8Non Churned Customers
We'll need to handle this when building our model


In [9]:
#Handling Missing Data
total_missing_values=df.isnull().sum().sum()
if total_missing_values>0:
  missing_by_col=df.isnull().sum()
  missing_cols=missing_by_col[missing_by_col>0]

  for col,count in missing_cols:
    percentage=(count/len(df)*100)
    print(f'{col}:{count}missing({percentage:.2f}%)')

else:
  print('There is No Missing Values')


empty_strings=(df['TotalCharges']==' ').sum()
print(f'The Empty Strings in the Total Charges:{empty_strings}')

There is No Missing Values
The Empty Strings in the Total Charges:11


In [10]:
print(df['gender'].unique())
print(df['Contract'].unique())


print(df.groupby('gender')['Churn'].value_counts())
print(df.groupby('Contract')['MonthlyCharges'].value_counts())

print(df['PaymentMethod'].value_counts())
print(df['InternetService'].value_counts())

print(df[df['tenure']>60])
print(df[(df['Churn']=='Yes')& (df['SeniorCitizen']==1)])


['Female' 'Male']
['Month-to-month' 'One year' 'Two year']
gender  Churn
Female  No       2549
        Yes       939
Male    No       2625
        Yes       930
Name: count, dtype: int64
Contract        MonthlyCharges
Month-to-month  20.05             28
                20.00             20
                20.25             19
                19.65             18
                20.35             18
                                  ..
Two year        117.60             1
                118.20             1
                118.60             1
                118.65             1
                118.75             1
Name: count, Length: 2892, dtype: int64
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
      customerID  gender  SeniorCitizen Partner Dependent

In [11]:
internet_services=df['InternetService'].value_counts()
for service,count in internet_services.items():
  print(f'{service}:{count} Customers with ({count/len(df)*100:.1f}%)')

senior=df.groupby('SeniorCitizen')['Churn'].value_counts(normalize=True)
print(senior)

payment_counts=df['PaymentMethod'].value_counts().head(5)
for payment,count in payment_counts.items():
  print(f'{payment}:{count}')

contract_charges=df.groupby('Contract')['MonthlyCharges'].mean()
for contract,avg_charge in contract_charges.items():
  print(f'{contract}:{avg_charge}')

Fiber optic:3096 Customers with (44.0%)
DSL:2421 Customers with (34.4%)
No:1526 Customers with (21.7%)
SeniorCitizen  Churn
0              No       0.763938
               Yes      0.236062
1              No       0.583187
               Yes      0.416813
Name: proportion, dtype: float64
Electronic check:2365
Mailed check:1612
Bank transfer (automatic):1544
Credit card (automatic):1522
Month-to-month:66.39849032258064
One year:65.04860828241684
Two year:60.770412979351036
